# Water permittivity for ePCP-SAFT

Parameters of the permittivity model of Rueben et al. (2024a) for water with the liquid density of ePCP-SAFT
(Esper et al. 2023 water; identical to PCP-SAFT without ions). The permittivity is evaluated by feos (`State.permittivity`).

- Data: DDB, filter of Rueben et al. (2024a): liquid, frequency < 1000 Hz or not given, p < 5 bar or not given; all data points.
- Pressure: given pressure, else p = max(1 bar, psat of the model) (Rueben et al. 2024a).
- Loss: Huber, delta = 5 % of the mean permittivity (Rueben et al. 2024a).
- Scheme: P (a11mu2, a2), fixed by decision; NP and All are shown for information.
- Validation: IAPWS R8-97 with IAPWS-95 density, 273.16-498.15 K.

Requires `EPCPSAFT_DDB_PERMITTIVITY` (path of the DDB export). Outputs are derived from licensed data: run with
`jupyter nbconvert --execute --to notebook --output-dir results 01_water_permittivity.ipynb`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import permittivity as pm, iapws

CAS = "7732-18-5"
record = pm.esper_records()[CAS]
eos = pm.eos(record)
df = pm.rueben_filter(pm.ddb())
print("parameters/epcpsaft/esper2023.json:", record["permittivity_record"])

## Data and densities

In [ ]:
d = pm.substance_data(df, CAS)
d["rho"] = pm.liquid_density(eos, d["T"].values, d["p"].values)  # mol/m3
print(f"n = {len(d)}, T = {d['T'].min()}-{d['T'].max()} K, pressure given: {d['p'].notna().sum()}")

## Fits

In [ ]:
fits = {s: pm.fit(record, d["T"].values, d["rho"].values, d["eps"].values, s) for s in ["P", "NP", "All"]}
pd.DataFrame({s: {**dict(zip(pm.NAMES, f["x"])), "AICc": f["aicc"], **f["metrics"]} for s, f in fits.items()}).T.round(4)

Deviation of the fitted scheme-P parameters from the values in `parameters/epcpsaft/esper2023.json` (rounded to 4 decimals):

In [ ]:
x_file = np.array([record["permittivity_record"]["PerturbationTheory"][k] for k in pm.NAMES])
pd.Series(fits["P"]["x"] - x_file, index=pm.NAMES)

## Validation with IAPWS R8-97
Grid: 5 K steps at p = 0.1 MPa (T < 372 K) or 1.5 psat, and 10 MPa every 10 K.

In [ ]:
Ts = np.r_[273.16, np.arange(278.15, 498.16, 5.0)]
grid = pd.DataFrame([(t, iapws.default_pressure(t)) for t in Ts] + [(t, 1e7) for t in Ts[::2]], columns=["T", "p"])
grid["eps_iapws"] = [iapws.eps_r(t, p) for t, p in zip(grid["T"], grid["p"])]
grid["rho"] = pm.liquid_density(eos, grid["T"].values, grid["p"].values)
for s in ["P", "All"]:
    grid[s] = pm.permittivity(record, fits[s]["x"], grid["T"].values, grid["rho"].values)
pd.DataFrame({s: pm.metrics(grid[s], grid["eps_iapws"]) for s in ["P", "All"]}).T.round(4)

Deviation of the DDB data from IAPWS (same pressures as in the fit; IAPWS density):

In [ ]:
p_iapws = [max(1e5, iapws.psat(t)) if np.isnan(p) else p for t, p in zip(d["T"], d["p"])]
d["eps_iapws"] = [iapws.eps_r(t, p) for t, p in zip(d["T"], p_iapws)]
pd.Series(pm.metrics(d["eps"], d["eps_iapws"])).round(4)

## Relative deviations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for s, c in [("P", "C0"), ("All", "C1")]:
    ax[0].plot(d["T"], 100 * fits[s]["dev"] / d["eps"], ".", ms=3, color=c, label=s)
    g = grid[grid["p"] < 1e7]
    ax[1].plot(g["T"], 100 * (g[s] / g["eps_iapws"] - 1), "-", color=c, label=f"{s}: 0.1 MPa / 1.5 psat")
    g = grid[grid["p"] == 1e7]
    ax[1].plot(g["T"], 100 * (g[s] / g["eps_iapws"] - 1), "--", color=c, label=f"{s}: 10 MPa")
ax[1].plot(d["T"], 100 * (d["eps"] / d["eps_iapws"] - 1), ".", ms=2, color="0.6", label="DDB data")
ax[0].set_ylabel(r"$100\,(\varepsilon_\mathrm{model} - \varepsilon_\mathrm{ref}) / \varepsilon_\mathrm{ref}$")
ax[0].set_title("reference: DDB data"); ax[1].set_title("reference: IAPWS R8-97")
for a in ax:
    a.axhline(0, color="k", lw=0.5); a.set_xlabel("T / K"); a.set_ylim(-3, 3); a.legend(fontsize=7)
plt.tight_layout()

Points outside ±3 % (not shown):

In [ ]:
out = d[np.abs(fits["P"]["dev"] / d["eps"]) > 0.03]
out.assign(rel_dev_percent=100 * fits["P"]["dev"][out.index] / out["eps"])